In [1]:
import pandas as pd
import glob
import re
import calendar

In [2]:
# --- Process CPO Production data ---
cpo_files = glob.glob('../raw_data/PRODUCTION OF CRUDE PALM OIL*.xlsx')

In [3]:
def extract_malaysia_monthly_timeseries(excel_file):
    df_raw = pd.read_excel(excel_file, header=None, dtype=object)
    ncols = df_raw.shape[1]

    month_abbrs = {m[:3].upper() for m in calendar.month_name if m}

    def is_year_like(x):
        try:
            xi = int(float(x))
            return 1900 <= xi <= 2100
        except Exception:
            return False

    # find rows that look like month header rows
    month_rows = []
    for i, row in df_raw.iterrows():
        matches = 0
        for val in row:
            if pd.isna(val):
                continue
            s = str(val).strip()
            if len(s) >= 3 and s[:3].upper() in month_abbrs:
                matches += 1
        if matches >= 2:
            month_rows.append(i)

    if not month_rows:
        raise ValueError("No month header rows detected. Check sheet layout.")

    # find row where first column == "MALAYSIA" (case-insensitive, strict)
    malaysia_mask = df_raw.iloc[:, 0].astype(str).str.strip().str.upper() == "MALAYSIA"
    malaysia_idxs = df_raw.index[malaysia_mask].tolist()
    if not malaysia_idxs:
        raise ValueError("Row 'MALAYSIA' not found in first column.")
    malaysia_idx = malaysia_idxs[0]

    records = []
    def find_year_pair(yrow, start_col):
        candidates = [(start_col, start_col+1), (start_col-1, start_col)]
        for c1, c2 in candidates:
            if 0 <= c1 < ncols and 0 <= c2 < ncols:
                if is_year_like(df_raw.iat[yrow, c1]) and is_year_like(df_raw.iat[yrow, c2]):
                    return c1, c2
        return None

    for mrow in month_rows:
        # locate year row near month header
        yr_row = None
        for r in range(mrow+1, min(mrow+4, len(df_raw))):
            if sum(is_year_like(x) for x in df_raw.iloc[r]) >= 2:
                yr_row = r
                break
        if yr_row is None:
            continue

        for col, val in enumerate(df_raw.iloc[mrow]):
            if pd.isna(val):
                continue
            s = str(val).strip()
            if len(s) >= 3 and s[:3].upper() in month_abbrs:
                month_abbr = s[:3].title()
                pair = find_year_pair(yr_row, col)
                if not pair:
                    continue
                c1, c2 = pair
                try:
                    y1 = int(float(df_raw.iat[yr_row, c1]))
                    y2 = int(float(df_raw.iat[yr_row, c2]))
                except Exception:
                    continue
                v1 = pd.to_numeric(df_raw.iat[malaysia_idx, c1], errors='coerce')
                v2 = pd.to_numeric(df_raw.iat[malaysia_idx, c2], errors='coerce')
                records.append({"Year": y1, "Month": month_abbr, "Value": v1})
                records.append({"Year": y2, "Month": month_abbr, "Value": v2})

    df_rec = pd.DataFrame(records)
    if df_rec.empty:
        raise ValueError("No Malaysia values found.")

    df_rec["Date"] = pd.to_datetime(
        df_rec["Year"].astype(int).astype(str) + "-" + df_rec["Month"],
        format="%Y-%b", errors="coerce"
    )
    df_rec = df_rec.dropna(subset=["Date"])
    df_rec["Period"] = df_rec["Date"].dt.to_period("M")

    ts = df_rec.groupby("Period")["Value"].first().sort_index()
    # df_ts = ts.to_frame(name="FFB_Price")
    return ts

In [5]:
all_cpo_data = []
for file in cpo_files:
    data = extract_malaysia_monthly_timeseries(file)
    all_cpo_data.append(data)
df_all_cpo = pd.concat(all_cpo_data)
df_all_cpo = df_all_cpo[~df_all_cpo.index.duplicated(keep="first")].sort_index()
df_all_cpo

Period
2022-01    1253581
2022-02    1137462
2022-03    1411223
2022-04    1462053
2022-05    1461029
2022-06    1545111
2022-07    1253581
2022-08    1137462
2022-09    1411223
2022-10    1462053
2022-11    1461029
2022-12    1545111
2023-01    1380410
2023-02    1253573
2023-03    1288354
2023-04    1194559
2023-05    1517634
2023-06    1447697
2023-07    1380410
2023-08    1253573
2023-09    1288354
2023-10    1194559
2023-11    1517634
2023-12    1447697
2024-01    1402355
2024-02    1259382
2024-03    1392471
2024-04    1501905
2024-05    1704495
2024-06    1615283
2024-07    1402355
2024-08    1259382
2024-09    1392471
2024-10    1501905
2024-11    1704495
2024-12    1615283
2025-01    1239895
2025-02    1188442
2025-03    1387616
2025-04    1686379
2025-05    1771621
2025-06    1692310
2025-07    1239895
2025-08    1188442
2025-09    1387616
2025-10    1686379
2025-11    1771621
2025-12    1692310
Freq: M, Name: Value, dtype: int64

In [6]:
df_all_cpo.to_csv('../data/cpo_monthly.csv')

In [7]:
oer_files = glob.glob('../raw_data/OIL EXTRACTION RATE FOR CRUDE PALM OIL*.xlsx')
all_oer_data = []
for file in oer_files:
    data = extract_malaysia_monthly_timeseries(file)
    all_oer_data.append(data)
df_all_oer = pd.concat(all_oer_data)
df_all_oer = df_all_oer[~df_all_oer.index.duplicated(keep="first")].sort_index()
df_all_oer

Period
2022-01    19.68
2022-02    19.39
2022-03    19.57
2022-04    19.83
2022-05    19.98
2022-06    19.66
2022-07    19.68
2022-08    19.39
2022-09    19.57
2022-10    19.83
2022-11    19.98
2022-12    19.66
2023-01    19.71
2023-02    19.24
2023-03    19.60
2023-04    20.00
2023-05    20.01
2023-06    19.83
2023-07    19.71
2023-08    19.24
2023-09    19.60
2023-10    20.00
2023-11    20.01
2023-12    19.83
2024-01    19.37
2024-02    19.84
2024-03    19.80
2024-04    19.76
2024-05    19.52
2024-06    19.24
2024-07    19.37
2024-08    19.84
2024-09    19.80
2024-10    19.76
2024-11    19.52
2024-12    19.24
2025-01    19.23
2025-02    19.05
2025-03    19.15
2025-04    19.55
2025-05    19.89
2025-06    19.80
2025-07    19.23
2025-08    19.05
2025-09    19.15
2025-10    19.55
2025-11    19.89
2025-12    19.80
Freq: M, Name: Value, dtype: float64

In [8]:
df_all_oer.to_csv('../data/oer_monthly.csv')